# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

ML task type: Binary classification (provisional).

My initial task is to investigate whether historical content performance can help identify pages at risk of a meaningful future decline. The intended decision is to prioritize pages for review by the SEO or content team. I will initially frame this as binary classification: predicting whether a content item experiences a meaningful decline during a future window. This framing is provisional because the available starter dataset contains aggregated performance windows, and the feasibility of constructing a leakage-safe future target still needs to be verified.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proposed target: Meaningful future decline.**

The intended target is a binary label indicating whether a content item experiences a predefined meaningful decline in performance over a future 28-day window, based on a suitable performance measure such as impressions.

- `1`: The content item meets the predefined decline threshold during the future window.
- `0`: The content item does not meet the decline threshold.

The decline threshold and comparison baseline must be defined before creating the labels. Historical performance features must be calculated exclusively from data available before the prediction point. The current starter dataset's `trend_direction` and `trend_pct` columns will not be treated as verified future targets until their calculation windows are understood.

If the available data cannot support this target, I will document the limitation and refine the task before modeling.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@K.**

The main operational goal is to identify a useful shortlist of pages for human review. Precision@K will measure the proportion of the top K pages flagged by the model that actually experience the defined future decline.

For example, if the team reviews the 20 highest-risk pages, Precision@20 measures how many of those 20 pages meet the future-decline definition.

Recall will also be considered to assess how many of all declining pages the model identifies. Evaluation must use a time-aware validation setup, with features restricted to information available before the target window. The final metric and value of K should reflect the team's actual review capacity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: One content item (page).**

Each row represents one content item identified by `content_id`. The starter dataset contains 30,000 content items across 32 clients. I will inspect a small set of identifiers and historical performance fields to understand the available unit of analysis.

The displayed rows represent the current snapshot, not separate future prediction examples. A future target column will only be created after the temporal boundaries and availability of the underlying data have been verified.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

data_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

analysis_df = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "impressions_last_30d",
        "impressions_prev_30d",
    ]
].copy()

print(f"Rows: {len(analysis_df):,}")
print(f"Unique content items: {analysis_df['content_id'].nunique():,}")

analysis_df.head(10)

Rows: 30,000
Unique content items: 30,000


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,impressions_last_30d,impressions_prev_30d
0,content_304f48230142,client_f369cb89fc,3803,29,17,578,987
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,2501,5915
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,2382,6089
3,content_331d6c4de07b,client_19581e27de,11751,58,78,3626,4206
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,4211,6452
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,617,1009
6,content_9a34b442b552,client_8722616204,20,0,1,1,13
7,content_a63219c6e95a,client_19581e27de,1724,1,28,636,632
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,5696,13828
9,content_c27558df2b0c,client_19581e27de,1240,2,3,252,356


The intended target column is `future_decline`, with values of `1` or `0`. It is not created from the current `trend_direction` or `trend_pct` columns because their temporal definitions have not yet been verified. The following code sketches the target schema using missing values to indicate that the labels have not been constructed. These missing values are intentional placeholders, not observed labels.

In [2]:
target_sketch = analysis_df[["content_id", "client_id"]].copy()

target_sketch["future_decline"] = pd.Series(
    pd.NA,
    index=target_sketch.index,
    dtype="Int64",
)

print(target_sketch.head(10))
print("\nTarget label counts:")
print(target_sketch["future_decline"].value_counts(dropna=False))

             content_id          client_id  future_decline
0  content_304f48230142  client_f369cb89fc            <NA>
1  content_a1fb4e703a9e  client_4e07408562            <NA>
2  content_9aa793d4d895  client_7f2253d7e2            <NA>
3  content_331d6c4de07b  client_19581e27de            <NA>
4  content_d99b7a2d90ca  client_3fdba35f04            <NA>
5  content_d4084a4bc775  client_f369cb89fc            <NA>
6  content_9a34b442b552  client_8722616204            <NA>
7  content_a63219c6e95a  client_19581e27de            <NA>
8  content_5e6c160719bc  client_6208ef0f77            <NA>
9  content_c27558df2b0c  client_19581e27de            <NA>

Target label counts:
future_decline
<NA>    30000
Name: count, dtype: Int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule, such as flagging every page whose recent impressions have dropped by a chosen percentage, is a useful baseline. However, a single threshold may not capture different performance patterns across content items with different traffic levels, ages, and historical behavior.

Machine learning may help combine multiple historical signals to rank pages by future-decline risk. Its value must be demonstrated against a simple fixed-rule baseline using leakage-safe validation and an appropriate metric such as Precision@K.

ML is not automatically better than a rule. If it does not provide a meaningful improvement, the simpler and more interpretable rule may be the better operational choice. The intended output is a review-priority list for human decision-making, not an automatic content change.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.